In [2]:
from os import getcwd
cwd_os = getcwd()
print(f"working dir: {cwd_os}")


working dir: c:\SAVES\godot4\evo-3\script\python


single generation turn

In [ ]:
# https://proproprogs.ru/ga/ga-delaem-geneticheskiy-algoritm-dlya-zadachi-onemax

from sqlite3 import connect
from os import path
from random import seed, random, randint, shuffle


POPULATION_SIZE = 10
P_CROSSOVER = 0.9
P_MUTATION = 0.1
RANDOM_SEED = 42
seed(RANDOM_SEED)

db_path = '../../data/train.db'
if not path.exists(db_path):
	raise Exception("db not exists")

with connect(db_path, autocommit=True) as conn:
	cursor = conn.cursor()

	cursor.execute(f'''
		select s.id, s.fitness, w.joint, w.range
		from (
			select s.id, s.fitness
			from session s
			where s.fitness > 0
			order by s.ctime desc, s.id
			limit {POPULATION_SIZE}
		) s
		join walk_param w on s.id = w.session_id		
	''')

	sessions = []
	session_id = -1
	for id, fitness, joint, range_ in cursor.fetchall():
		if session_id != id:
			session = { "id": id, "fitness": fitness, "params": { joint: range_ } }
			session_id = id
			sessions.append(session)
		else:
			session["params"][joint] = range_

	pop_size = len(sessions)
	if pop_size % 2 > 0:
		sessions.pop()
		pop_size -= 1
	print(sessions)

	max_pop_id = pop_size - 1
	print(max_pop_id)

	param_keys = list(sessions[0]["params"].keys())
	print(param_keys)

	max_params = len(param_keys) - 1
	print(max_params)
	
	max_fitness = max(map(lambda x: x["fitness"], sessions))
	print(max_fitness)


	# selection
	offspring = []
	for _ in range(pop_size):
		i1 = i2 = i3 = 0
		while i1 == i2 or i1 == i3 or i2 == i3:
			i1, i2, i3 = randint(0, max_pop_id), randint(0, max_pop_id), randint(0, max_pop_id)
		offspring.append(max([sessions[i1], sessions[i2], sessions[i3]], key=lambda ind: ind["fitness"]))
	print(offspring)


	# crossover
	for i in range(0, pop_size, 2):
		if random() < P_CROSSOVER:
			child1 = offspring[i]
			child2 = offspring[i + 1]
			# print(i)
			# print(child1, child2)
			joint = param_keys[randint(0, max_params)]
			child1["params"][joint], child2["params"][joint] = child2["params"][joint], child1["params"][joint]
			# print(child1, child2)


	# mutation
	for mutant in offspring:
		if random() < P_MUTATION:
			joint = param_keys[randint(0, max_params)]
			mutant["params"][joint] = random()


	# save to DB
	for session in offspring:
		cursor.execute()
		# TODO





[{'id': 91, 'fitness': 0.5050423290540605, 'params': {'foot': 0.12245971806268714, 'hip_L': 0.17532560658662158, 'calf_L': 0.5927594721781311, 'hip_R': 0.9616060515686871, 'calf_R': 0.1396981146712859}}, {'id': 92, 'fitness': 0.5050423290540605, 'params': {'foot': 0.45864680565436755, 'hip_L': 0.014826005116043232, 'calf_L': 0.632650712094722, 'hip_R': 0.9590897285247629, 'calf_R': 0.0007252491930147506}}, {'id': 93, 'fitness': 0.5075644228007361, 'params': {'foot': 0.48162349766187373, 'hip_L': 0.02286173841062511, 'calf_L': 0.4210466418829072, 'hip_R': 0.9191195482721639, 'calf_R': 0.01390531661277366}}, {'id': 94, 'fitness': 0.5050423290540605, 'params': {'foot': 0.36379690215473837, 'hip_L': 0.1056586480712031, 'calf_L': 0.49179975500078366, 'hip_R': 0.9428510932109669, 'calf_R': 0.13286829323327606}}, {'id': 95, 'fitness': 0.5100836325993666, 'params': {'foot': 0.36490781641198633, 'hip_L': 0.1615476938140296, 'calf_L': 0.38380581140407205, 'hip_R': 0.8974122636155789, 'calf_R': 0